# Retail Customer-Month Dataset Exploration
One row represents one customer at one monthly snapshot. Historical features use only information before the snapshot; targets describe the month beginning at the snapshot.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

candidate = Path.cwd()
PROJECT_ROOT = candidate if (candidate / 'data').exists() else candidate.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'customer_monthly_ml.csv'
data = pd.read_csv(DATA_PATH, parse_dates=['snapshot_date'])
data.head()

## Shape, types, missing values, and time coverage
Blank `days_since_last_purchase` means that no earlier purchase exists; it is not automatically a bad record.

In [ ]:
print('Shape:', data.shape)
display(data.dtypes.to_frame('dtype'))
display(data.isna().sum().to_frame('missing'))
print('Snapshots:', data['snapshot_date'].min(), 'to', data['snapshot_date'].max())
display(data['snapshot_date'].value_counts().sort_index().to_frame('rows'))

## Targets
The regression target keeps zero-spend customers. The classification target is 1 exactly when target-month spend is positive.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=data, x='will_purchase_next_month', ax=axes[0])
axes[0].set_title('Next-month purchase class')
sns.histplot(data=data, x='next_month_spend_lkr', bins=50, ax=axes[1])
axes[1].set_title('Next-month spend (LKR)')
plt.tight_layout()
plt.show()

## Historical feature distributions and correlation

In [ ]:
FEATURES = [
    'tenure_days', 'orders_last_90d', 'spend_last_90d_lkr',
    'avg_order_value_last_90d_lkr', 'order_value_std_last_90d_lkr',
    'days_since_last_purchase', 'has_previous_purchase'
]
data[FEATURES].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 7))
sns.heatmap(data[FEATURES].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Historical-feature correlation')
plt.tight_layout()
plt.show()

## Repeated customers and chronological evaluation
Customers occur in multiple snapshots, so random row splitting could mix future and past rows. The project trains on 2025-04 through 2025-11, validates on 2025-12 through 2026-01, and tests on 2026-02 through 2026-03. This estimates rolling future outcomes for existing customers, not entirely unseen customers.

Model inputs use only the seven historical features above. `customer_id` and `snapshot_date` are metadata. Both `next_month_spend_lkr` and `will_purchase_next_month` are targets and both must be excluded from X to prevent leakage.

In [ ]:
print('Unique customers:', data['customer_id'].nunique())
print('Rows per customer summary:')
display(data.groupby('customer_id').size().describe())
display(data.groupby('snapshot_date')[['next_month_spend_lkr', 'will_purchase_next_month']].agg(['mean', 'median']))